# Module 4, Section 2: Deep Agents 201: Harness Engineering

Section 1 introduced the Deep Agents building blocks with a pricing research agent. In Module 1 we hand-built a supervisor that routed customer questions to a database agent and a documents agent. Now we rebuild **the same TechHub support agent on the Deep Agents harness**, then improve it by tuning the harness while the model stays fixed.

> **Harness engineering**: the harness is everything around the model: the system prompt, tools, context, middleware, execution environment, and verification. You find a failure mode in a trace, change the harness, and re-verify. Done this way, LangChain's coding agent went from 52.8% to 66.5% on Terminal Bench 2.0 (from outside the Top 30 to the Top 5) with the model held fixed ([Improving Deep Agents with harness engineering](https://www.langchain.com/blog/improving-deep-agents-with-harness-engineering)).

In this section:
1. **Same job, new harness**: the Module 1 supervisor as a deep agent
2. **Context engineering**: `AGENTS.md` vs skills vs memory
3. **Routing memory** with `CompositeBackend` (per-customer memory, read-only org policy)
4. **Context quarantine**: why subagents keep the main context small
5. **Custom middleware**: verified identity, PII redaction, and a customer-scope guard
6. **Human-in-the-loop** on a sensitive tool (refunds)
7. **Read-only org assets** with filesystem permissions

Section 3 covers dynamic subagents and verification loops.

## Setup

**Prerequisites:** `uv sync`, a `.env` with model credentials (see `.env.example`: your own OpenAI and/or Anthropic key, or the LLM Gateway), and the document vectorstore, which the product-policy specialist searches: `uv run python data/data_generation/build_vectorstore.py` (one-time, about 60 seconds).

In [ ]:
import uuid
import warnings

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
warnings.filterwarnings("ignore", category=UserWarning)

from utils.langsmith_links import project_url, trace_link

print(f"📊 All traces from this workshop: {project_url()}")

In [ ]:
def new_thread():
    """A fresh conversation (thread) for each example."""
    return {"configurable": {"thread_id": str(uuid.uuid4())}}


def tool_calls(result):
    """Names of the tools the main agent called (subagent name in brackets for `task`)."""
    names = []
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", None) or []:
            extra = f"[{tc['args'].get('subagent_type')}]" if tc["name"] == "task" else ""
            names.append(tc["name"] + extra)
    return names


def show(result, width=600):
    print("🔧 Tool calls:", tool_calls(result))
    print("💬 Answer:", result["messages"][-1].text[:width])

## 1. Same job, new harness

A deep agent is `create_agent()` plus a pre-built middleware stack: a virtual **filesystem**, **subagents** through a `task` tool, **context management** (offloading large tool results and summarizing long conversations), and optional planning.

Below is the Module 1 supervisor rebuilt as a deep agent. The specialists become **subagent specs** (name, description, prompt, tools) instead of hand-wrapped tool functions.

In [ ]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

from agents.db_agent import DB_AGENT_BASE_TOOLS
from agents.docs_agent import DOCS_AGENT_BASE_TOOLS
from config import DEEP_AGENT_MODEL

llm = init_chat_model(DEEP_AGENT_MODEL)  # openai:gpt-5.4 by default (config.py)

simple_agent = create_deep_agent(
    model=llm,
    system_prompt="You are TechHub's customer support agent. Delegate to your specialists and answer concisely.",
    subagents=[
        {
            "name": "order-specialist",
            "description": "Looks up TechHub orders, order items, prices, and product availability.",
            "system_prompt": "You are TechHub's order specialist. Report exactly what the tools return.",
            "tools": DB_AGENT_BASE_TOOLS,
        },
        {
            "name": "product-policy-specialist",
            "description": "Searches TechHub product specs and store policies (returns, warranty, shipping).",
            "system_prompt": "You are TechHub's product and policy specialist. Quote the terms that answer the question.",
            "tools": DOCS_AGENT_BASE_TOOLS,
        },
    ],
)

# The harness gives the agent these tools before we add any of our own
print("Built-in tools:", sorted(simple_agent.nodes["tools"].bound.tools_by_name))

In [ ]:
with trace_link("Deep agent: same question as Module 1"):
    result = simple_agent.invoke(
        {"messages": [{"role": "user", "content": "How much is TECH-LAP-001 and does it have a warranty?"}]}
    )
show(result)

**What changed versus Module 1:** we didn't write any tool-wrapping code. The `task` tool delegates to a subagent, which runs in its own context window and returns only its final answer.

`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, and `delete` work on the agent's filesystem. `execute` only runs commands when the backend is a sandbox, which we add in Module 6.

From here on we use the full TechHub harness from `agents/deep_support_agent.py`, built step by step below.

## 2. Context engineering: AGENTS.md vs skills vs memory

A deep agent has three ways to give the model knowledge, each loaded differently:

| Surface | Loaded | Use for | TechHub file |
|---|---|---|---|
| **`AGENTS.md` (memory)** | Always, injected into the system prompt | Operating rules every turn needs | `agents/deep_support/AGENTS.md` |
| **Skills** | On demand: name and description at startup, the body only when a task matches | Procedures for specific tasks | `agents/deep_support/skills/*/SKILL.md` |
| **Long-term memory files** | When the agent reads them | Things learned about a customer over time | `/memories/preferences.md` (per customer) |

Skills use **progressive disclosure**: only the frontmatter costs context until the skill is actually needed.

In [ ]:
from pathlib import Path

harness_dir = Path("../../agents/deep_support")
for path in sorted(harness_dir.rglob("*.md")):
    print(path.relative_to(harness_dir))

print("\n--- skills/returns-and-warranty/SKILL.md (frontmatter) ---")
print("\n".join((harness_dir / "skills/returns-and-warranty/SKILL.md").read_text().splitlines()[:4]))

In [ ]:
from langgraph.store.memory import InMemoryStore

from agents import SupportContext, create_deep_support_agent

store = InMemoryStore()  # holds /memories/ so we can inspect it below
agent = create_deep_support_agent(store=store)

with trace_link("Skill loaded on demand"):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "I opened my new laptop 10 days ago and don't like it. Can I return it?"}]},
        config=new_thread(),
        context=SupportContext(customer_id="CUST-003"),
    )
show(result)

Look at the tool calls: the agent used `read_file` to open the matching `SKILL.md` and then followed its steps. Open the trace to see which files it read.

**Context management is on by default.** Large tool results are offloaded to the filesystem and replaced with a file path and a short preview. When the conversation gets close to the model's context limit, older turns are summarized. See [Deep Agents context engineering](https://docs.langchain.com/oss/python/deepagents/overview) for the current thresholds.

## 3. Routing memory with `CompositeBackend`

A `CompositeBackend` routes filesystem paths to different storage backends. The TechHub agent uses three routes:

| Path | Backend | Scope |
|---|---|---|
| `/harness/` | `FilesystemBackend` on `agents/deep_support/` | Org-owned `AGENTS.md`, skills, and policies (read-only, see §7) |
| `/memories/` | `StoreBackend`, namespaced by the **verified** customer ID | Long-term memory per customer |
| everything else | `StateBackend` | Scratch files for this thread only |

The namespace comes from the runtime context the *application* sets, never from what the user types. So one customer's memory can never reach another customer's conversation. With no verified customer, memory is scoped to the conversation itself: a shared "anonymous" bucket would let one unverified user plant notes that the next one loads.

In [ ]:
# Thread 1: CUST-003 states a lasting preference
r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Please remember: I prefer email updates and I only buy Logitech peripherals."}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-003"),
)
show(r1, 200)

# Thread 2: a brand-new conversation, same customer
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Which mouse would you recommend for me?"}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-003"),
)
show(r2, 300)

In [ ]:
# Thread 3: a different customer asks the same question
r3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Which mouse would you recommend for me?"}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-019"),
)
show(r3, 300)

print("\n🗂️ Memory namespaces in the store:", list(store.list_namespaces()))
for item in store.search(("techhub", "CUST-003")):
    print(f"\n{item.key}:")
    print(item.value.get("content") if isinstance(item.value, dict) else item.value)

CUST-003's second thread used the stored preference. CUST-019 has no memory, so the agent asked or answered generically.

> ⚠️ `CompositeBackend` strips the route prefix before handing the path to the backend, so `/memories/preferences.md` is stored under the key `/preferences.md`. Keep this in mind if you seed memory from application code.

## 4. Context quarantine with subagents

Each subagent runs in its own context window. The main agent sees only the subagent's final answer, not the dozens of tool calls behind it. That keeps the main context small, which means lower cost and latency, and fewer chances for the model to lose track.

We stream with `subgraphs=True` to count tool calls inside and outside the subagents.

In [ ]:
main_calls, subagent_calls = [], []
for namespace, chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What are my 5 most recent orders, with product names?"}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-003"),
    stream_mode="updates",
    subgraphs=True,
):
    for update in chunk.values():
        for m in (update or {}).get("messages", []) if isinstance(update, dict) else []:
            for tc in getattr(m, "tool_calls", None) or []:
                (subagent_calls if namespace else main_calls).append(tc["name"])

print(f"Main agent tool calls ({len(main_calls)}): {main_calls}")
print(f"Subagent tool calls   ({len(subagent_calls)}): {sorted(set(subagent_calls))}")

### Fewer tools per model call

Quarantine keeps the specialist's work out of the main context, but the specialist's own context still carries every tool schema on every call. The order specialist sees 12 tools: 7 built-in file tools from the harness and 5 order tools. Most questions need 2 or 3 of them.

LangChain has three levers for this ([built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in), [skills](https://docs.langchain.com/oss/python/deepagents/skills)):

| Lever | How it works | Cost |
|---|---|---|
| Provider tool search (`ProviderToolSearchMiddleware`) | Tools are deferred; the model searches the provider's index and loads a schema when it needs one | A search turn; needs OpenAI gpt-5.4+ or a recent Claude model |
| Skill tools (`include_tools` in `SKILL.md`) | A tool becomes callable only after the agent reads the skill that lists it | Fits when tools group by procedure |
| `LLMToolSelectorMiddleware` | A model picks the relevant tools before each model call | One extra model call per turn |

The general guidance is to start with provider tool search for large tool registries, and to use the LLM selector for models without it. But measure on your own agent. `create_deep_support_agent(narrow_tools=...)` adds either one to the order specialist, so we can compare them on the same question.

The selector's settings:

| Setting | Value | Why |
|---|---|---|
| `model` | unset | Selection uses the specialist's own (small) model, and switches with it on a fallback |
| `max_tools` | 3 | Extra selections beyond 3 are dropped |
| `always_include` | `["read_file"]` | The harness saves large tool results to files and reads them back with `read_file` |
| `system_prompt` | Select tools for **every step** of the task | Selection reruns before each model call, from the task message only |
| `on_parsing_failure` | `"all"` | Selection is an optimization; a malformed answer keeps every tool instead of failing the run |

The main agent gets neither: its 9 tools are mostly the harness's own (`task`, the file tools), and losing `task` would stop delegation.

In [ ]:
def specialist_usage(agent, question):
    """Input tokens across the specialists' model calls, and the tools they called."""
    tokens, calls = 0, []
    for namespace, chunk in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config=new_thread(),
        context=SupportContext(customer_id="CUST-003"),
        stream_mode="updates",
        subgraphs=True,
    ):
        for update in chunk.values():
            for m in (update or {}).get("messages", []) if isinstance(update, dict) else []:
                if namespace and m.type == "ai":
                    tokens += (m.usage_metadata or {}).get("input_tokens", 0)
                    calls += [tc["name"] for tc in m.tool_calls]
    return tokens, sorted(set(calls))

question = "What are my 5 most recent orders, with product names?"
variants = [
    ("All 12 tools", agent),
    ("Tool search", create_deep_support_agent(narrow_tools="search")),
    ("LLM selector", create_deep_support_agent(narrow_tools="selector")),
]
for label, a in variants:
    tokens, calls = specialist_usage(a, question)
    print(f"{label:13} specialist input tokens: {tokens:>6,}  tools called: {calls}")

What we measured on this specialist (2 to 3 runs per model; your numbers will vary):

| Order specialist | `gpt-5.4-mini` | Claude Haiku 4.5 |
|---|---|---|
| All 12 tools | ~10.9k tokens, correct | ~19k tokens, correct |
| Provider tool search | 14k to 35k tokens (+30% to 3x), correct | Missed the order tools in both runs: it searched the files instead |
| LLM selector | ~5.0k tokens (−54%), correct | ~10k tokens (−46%), correct |

With only 5 deferred tools, the search turn cost more than the schemas it saved, and the visible file tools drew Haiku away from searching. Provider tool search earns its place at 30+ tools. At this size the LLM selector wins. Things to weigh with the selector:

- **The selection is itself a model call**, one per specialist turn, so it adds latency (3 to 5 seconds per run here) and its own tokens, which the numbers above don't include. In LangSmith the selection call is tagged `lc_source: tool_selection`.
- **A changing tool list breaks the prompt cache.** Tool schemas sit at the top of the prompt. Here the selection stayed the same on every call of a task, so the prompt prefix stayed stable.
- **Selection reruns on every call and sees only the task message.** With the default prompt, Claude Haiku sometimes dropped `get_order_items` after the first lookup, and the answer lost its product names. The fix was the "every step" selection prompt. Run your evals with the selector on before you ship it.
- **Neither lever is an access control.** They narrow what the model sees, and the full tool list can still execute. The selector is an LLM reading user text, so it can be talked into choosing a tool. Authorization stays in `customer_scope_guard`, which checks every call either way.

## 5. Custom middleware: identity, privacy, and guardrails

Middleware wraps the model call and the tool call, so you can add logic without touching agent code. The TechHub harness adds these pieces, in `agents/middleware/` and the factory:

| Middleware | Hook | What it does |
|---|---|---|
| `PIIMiddleware("email", strategy="redact", ...)` | input, output, tool results | Emails become `[REDACTED_EMAIL]` in what the customer types, what the model replies, and what tools return |
| `PIIMiddleware("credit_card", strategy="mask", ...)` | input, output, tool results | Card numbers become `****-****-****-1234` |
| `verified_customer_prompt` | `wrap_model_call` | Tells the model which customer the *application* verified |
| `customer_scope_guard` | `wrap_tool_call` | Refuses any tool call outside the verified customer's own data |
| `fault_tolerance_middleware()` | model and tool calls | Retries, a fallback model, and per-run call limits (below) |

> **Why a tool-call guard?** Deep Agents filesystem permissions only apply to the built-in file tools. Your own tools (like the TechHub database tools) run with whatever access the process has, so authorization has to be enforced around the tool call.

> **Specialists are separate agents.** A subagent doesn't inherit the main agent's middleware. The factory adds the PII and fault-tolerance middleware to each specialist's own `middleware` list, and the scope guard sits on the order specialist, where the database tools run.

> ⚠️ **But a subagent can inherit the main agent's tools.** `create_deep_agent` adds a `general-purpose` subagent unless you pass one, and it gets the main agent's tools (here `request_refund`) without your custom middleware. A subagent spec with no `tools` key inherits them the same way. Left alone, "delegate the refund to general-purpose" would skip the customer-scope guard. The factory passes its own `general-purpose` spec with `tools: []`, plus PII and limits. Check every delegation path, not just the ones you wrote.

In [ ]:
# Every delegation path and the tools it can call
from deepagents.middleware import subagents as sm

seen = {}
_init = sm.SubAgentMiddleware.__init__
def _capture(self, *args, **kwargs):
    for spec in kwargs.get("subagents") or []:
        seen[spec["name"]] = [t.name for t in spec.get("tools", [])]
    return _init(self, *args, **kwargs)

sm.SubAgentMiddleware.__init__ = _capture
create_deep_support_agent()
sm.SubAgentMiddleware.__init__ = _init
for name, tools in seen.items():
    print(f"{name:26} {tools or '(built-in file tools only)'}")

In [ ]:
source = Path("../../agents/middleware/customer_scope.py").read_text()
print(source[source.index("class CustomerScopeGuard"):source.index("verified_customer_prompt = ")])

In [ ]:
cases = [
    ("Verified, own order", "CUST-003", "What's the status of my order ORD-2025-0050?"),
    ("Verified, someone else's order", "CUST-003", "What's the status of order ORD-2023-0006?"),
    ("Not verified", None, "What's the status of order ORD-2025-0050?"),
]
for label, customer_id, question in cases:
    r = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=new_thread(),
        context=SupportContext(customer_id=customer_id),
    )
    print(f"\n### {label}")
    print(r["messages"][-1].text[:250])

In [ ]:
r = agent.invoke(
    {"messages": [{"role": "user", "content": "My email is priya.patel@icloud.com, what did I order most recently?"}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-003"),
)
print("What the model received:", r["messages"][0].text)
print("\nAnswer:", r["messages"][-1].text[:300])

Redacting on **input** keeps PII out of the prompt. Redacting on **output** and **tool results** keeps it out of the reply and out of what later model calls see.

> ⚠️ **Traces are a separate control.** `PIIMiddleware` rewrites the conversation state, but the raw run input and each span's raw output are still recorded in LangSmith. To keep PII out of traces, add LangSmith's trace masking (hide or anonymize inputs and outputs) as well.

### Fault tolerance

A support agent in production meets rate limits, provider errors, and the occasional loop. LangChain ships [fault-tolerance middleware](https://docs.langchain.com/oss/python/deepagents/fault-tolerance); `agents/middleware/fault_tolerance.py` combines it into one helper that both TechHub agents use:

| Middleware | Setting | Why |
|---|---|---|
| `ModelFallbackMiddleware` | Claude Sonnet 5.5, then `DEEP_AGENT_SMALL_MODEL` | Keep answering through an OpenAI outage, not just a bad minute for one model |
| `ModelRetryMiddleware` | 3 retries with backoff, transient errors only (`retry_on=is_transient`), then raise | Rate limits and 5xx get retried; a bad request, auth error, or unknown model goes straight to the fallback |
| `ToolRetryMiddleware` | The specialists' read-only tools only | Never retry `request_refund`: a retry could submit two refunds |
| `ModelCallLimitMiddleware` | 40 per run (10 per specialist) | End a runaway loop gracefully |
| `ToolCallLimitMiddleware` | 80 per run, `task` capped at 20 | Bound the delegations the model makes. Dynamic subagents dispatched from interpreter code aren't counted: `CodeInterpreterMiddleware(max_subagent_concurrency=4)` and each subagent's own limits bound those |

In [ ]:
print(Path("../../agents/middleware/fault_tolerance.py").read_text().split('"""', 2)[2])

### Cross-provider fallback

A fallback to a smaller model from the **same** provider doesn't help when the provider itself is down. So the first fallback is from the other provider: with OpenAI as the main model, `config.DEEP_AGENT_FALLBACK_MODELS` puts Claude Sonnet 5.5 ahead of `gpt-5.4-mini` (with Claude as the main model, `gpt-5.4` comes first). A model is included only if its credentials are set, or the LLM Gateway is on. With a single provider key, the chain is just the small model. Override the first entry with `DEEP_AGENT_FALLBACK_MODEL`.

Simulate an outage: point the main agent at a model name that doesn't exist on your main provider, so every call fails with a 404. `is_transient` doesn't retry a 404, so the fallback takes over at once. The cell prints the chain, so you can see which model should answer.

In [ ]:
from config import DEEP_AGENT_FALLBACK_MODELS

from config import DEEP_AGENT_MODEL

print("Main model:", DEEP_AGENT_MODEL, "| fallback chain:", DEEP_AGENT_FALLBACK_MODELS)

# Same provider as the main model, so its credentials work, but a model name that doesn't exist
outage_agent = create_deep_support_agent(model=f"{DEEP_AGENT_MODEL}-outage-simulation")

with trace_link("Fallback: main model down"):
    r = outage_agent.invoke(
        {"messages": [{"role": "user", "content": "What's the status of my most recent order?"}]},
        config=new_thread(),
        context=SupportContext(customer_id="CUST-003"),
    )

print("Main agent answered with:", {m.response_metadata.get("model_name") for m in r["messages"] if m.type == "ai"})
print("\n" + r["messages"][-1].text[:400])

Two things to know about cross-provider fallback:
- **It works mid-conversation.** The thread's earlier turns, including OpenAI tool calls, are converted to the Anthropic format on each call. We tested a gpt-5.4 turn followed by a Claude turn on the same thread.
- **Provider-specific tools don't carry over.** The Section 1 pricing agent uses its provider's built-in web search, which the other provider can't call, so that agent keeps a same-provider fallback.

## 6. Human-in-the-loop on sensitive tools

`interrupt_on={"request_refund": True}` pauses the run before `request_refund` executes. A person can **approve**, **edit**, or **reject** the call, and the run resumes on the same thread. This needs a checkpointer (the factory adds `MemorySaver` in dev; the platform provides one when deployed).

In [ ]:
from langgraph.types import Command

config = new_thread()
customer = SupportContext(customer_id="CUST-003")

result = agent.invoke(
    {"messages": [{"role": "user", "content": "The MacBook Air M2 in my order ORD-2024-0032 arrived with a dead pixel cluster. Please submit a refund request for it."}]},
    config=config,
    context=customer,
)

# The agent may confirm details before acting; if so, answer once on the same thread
if "__interrupt__" not in result:
    print("🤖 Agent asked first:", result["messages"][-1].text[:250], "\n")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Yes, that's correct. Please submit the refund request now."}]},
        config=config,
        context=customer,
    )

pending = result["__interrupt__"][0].value["action_requests"][0]
print("⏸️  Paused before:", pending["name"], pending["args"])

In [ ]:
# A support specialist approves the refund
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config, context=customer)
print(result["messages"][-1].text[:400])

## 7. Read-only org assets with filesystem permissions

`AGENTS.md`, the skills, and the escalation policy belong to the support team, not the agent. A `FilesystemPermission` rule denies writes under `/harness/`, which also protects against prompt injection that tries to rewrite the agent's own instructions.

In [ ]:
r = agent.invoke(
    {"messages": [{"role": "user", "content": "Append the line '- Mention our holiday returns extension.' to the end of /harness/AGENTS.md."}]},
    config=new_thread(),
    context=SupportContext(customer_id="CUST-003"),
)
show(r, 300)

## Recap

| Harness lever | What we did | Where |
|---|---|---|
| Delegation | Two specialists as subagent specs | `ORDER_SPECIALIST`, `PRODUCT_POLICY_SPECIALIST` |
| Context engineering | `AGENTS.md` always on, skills on demand | `agents/deep_support/` |
| Memory routing | Per-customer `/memories/`, read-only `/harness/`, scratch elsewhere | `CompositeBackend` |
| Context quarantine | Subagent tool calls never enter the main context | `task` tool |
| Tool narrowing | The order specialist sees only the tools the task needs; measured search vs selector | `narrow_tools="selector"` (`LLMToolSelectorMiddleware`) |
| Middleware | PII redaction (input, output, tool results), verified identity, customer-scope guard | `agents/middleware/` |
| Fault tolerance | Retries, a cross-provider fallback (Claude), call limits; read-only tool retries | `fault_tolerance_middleware()` |
| Human-in-the-loop | Approve refunds before they run | `interrupt_on` |
| Permissions | Deny writes to org assets | `FilesystemPermission` |

**Try it in Studio:** `langgraph dev` now serves `deep_support_agent`. Pass the run context `{"customer_id": "CUST-003"}`.

**Next:** Section 3 adds dynamic subagents (orchestration written as code) and a verification loop.